# 03 — Обучение трёх детекторов и тесты на обобщение

Вход: `dataset.zip` из ноутбука 02 (5000 картинок, `metadata.csv`, `splits/`).

Детекторы (real vs fake):
1. **SimpleCNN** — небольшая свёрточная сеть с нуля (базовый уровень);
2. **ResNet-50** — предобученная на ImageNet, дообучаем целиком;
3. **CLIP ViT-L/14 + logistic regression** — замороженные признаки CLIP, сверху линейный классификатор
   (подход «универсального детектора», Ojha et al. 2023).

Тесты: `test_seen` (генераторы из обучения), `test_unseen` (3 генератора, которых в обучении не было),
`test_own` (наши 2000: COCO real + SDXL-Turbo t2i/i2i). Метрики: accuracy, AUC, AP, доля пойманных фейков (TPR),
доля верно принятых real (TNR). Плюс таблица «по генераторам», матрицы ошибок, главный график и примеры ошибок.

**Runtime → GPU (T4)**: полный прогон ≈ 30–40 минут. Без GPU ноутбук сам переключается в облегчённый режим (меньше эпох, CLIP ViT-B/16), ≈ 2 часа на CPU. Результаты: `results/` → `results.zip` (и копия в Google Drive).

Контроль разрешения: фейки из CF в основном 256×256, real — крупнее. Чтобы детектор не учился «по размеру»,
каждая картинка сначала приводится к короткой стороне 256, потом вырезается 224×224 — для всех одинаково.

In [ ]:
!pip -q install scikit-learn transformers

In [ ]:
import os, io, json, time, random, zipfile, shutil
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

CFG = dict(
    DATASET_ZIP = ["/content/drive/MyDrive/deepfake_paper/dataset.zip", "/content/dataset.zip"],
    ROOT = "/content/dataset",
    RES  = "/content/results",
    SEED = 42, IMG = 224, BS = 32, WORKERS = 4,
    EPOCHS_CNN = 10,   LR_CNN = 1e-3,
    EPOCHS_RESNET = 5, LR_RESNET = 1e-4,
    CLIP_ID = "openai/clip-vit-large-patch14",
)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)
if DEV != "cuda":
    # Режим CPU (когда лимит GPU в Colab исчерпан): меньше эпох и лёгкая версия CLIP, чтобы уложиться в ~2 часа.
    # Для финальной версии статьи прогон на GPU с настройками по умолчанию даст полную конфигурацию (ViT-L/14, 10/5 эпох).
    CFG.update(EPOCHS_CNN=5, EPOCHS_RESNET=3, CLIP_ID="openai/clip-vit-base-patch16", WORKERS=2)
    torch.set_num_threads(os.cpu_count() or 2)
    print("GPU нет — включён облегчённый режим CPU:", {k: CFG[k] for k in ("EPOCHS_CNN", "EPOCHS_RESNET", "CLIP_ID")})
CLIP_DTYPE = torch.float16 if DEV == "cuda" else torch.float32
for d in (CFG["RES"], f"{CFG['RES']}/figs", f"{CFG['RES']}/models"):
    os.makedirs(d, exist_ok=True)

## 1. Данные

In [ ]:
try:
    from google.colab import drive; drive.mount("/content/drive")   # Drive нужен и для датасета, и для сохранения промежуточных результатов
except Exception as e:
    print("Drive не подключён:", e)
zpath = next((p for p in CFG["DATASET_ZIP"] if os.path.exists(p)), None)
assert zpath, "dataset.zip не найден: положите его в Google Drive/deepfake_paper или в /content/dataset.zip"
if not os.path.exists(f"{CFG['ROOT']}/metadata.csv"):
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CFG["ROOT"])
print("датасет:", zpath)

meta = pd.read_csv(f"{CFG['ROOT']}/metadata.csv", dtype=str).fillna("")
meta["y"] = (meta.label == "fake").astype(int)
meta["path"] = CFG["ROOT"] + "/images/" + meta.filename
# у CF-фейков стоит fake_type=t2i, хотя GAN'ы генерируют без текста — называем их просто generated
meta.loc[(meta.origin == "cf") & (meta.y == 1), "fake_type"] = "generated"
meta["held_out"] = pd.to_numeric(meta.held_out, errors="coerce").fillna(0).astype(int)

train_df = meta[meta.split == "train"].reset_index(drop=True)
val_df   = meta[meta.split == "val"].reset_index(drop=True)
# test_unseen содержит только фейки; чтобы считать AUC/accuracy, добавляем к нему real-картинки из test_seen
EVAL = {
    "test_seen":   meta[meta.split == "test_seen"].reset_index(drop=True),
    "test_unseen": pd.concat([meta[meta.split == "test_unseen"], meta[(meta.split == "test_seen") & (meta.y == 0)]]).reset_index(drop=True),
    "test_own":    meta[meta.split == "test_own"].reset_index(drop=True),
}
print(meta.groupby(["split", "label"]).size().unstack(fill_value=0))
print("\ntrain:", len(train_df), "val:", len(val_df), {k: len(v) for k, v in EVAL.items()})

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([T.Resize(256), T.RandomCrop(CFG["IMG"]), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([T.Resize(256), T.CenterCrop(CFG["IMG"]), T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(torch.utils.data.Dataset):
    def __init__(self, df, tf): self.p, self.y, self.tf = df.path.tolist(), df.y.values.astype(np.float32), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

def loader(df, tf, shuffle=False):
    return torch.utils.data.DataLoader(ImgDS(df, tf), batch_size=CFG["BS"], shuffle=shuffle,
                                       num_workers=CFG["WORKERS"], pin_memory=(DEV == "cuda"), persistent_workers=True)

## 2. Общие функции: обучение, предсказание, метрики

In [ ]:
def metrics(y, p, thr=0.5):
    y, p = np.asarray(y), np.asarray(p); pred = (p >= thr).astype(int)
    out = dict(n=len(y), acc=accuracy_score(y, pred))
    out["auc"] = roc_auc_score(y, p) if len(set(y)) == 2 else np.nan
    out["ap"]  = average_precision_score(y, p) if len(set(y)) == 2 else np.nan
    out["tpr"] = pred[y == 1].mean() if (y == 1).any() else np.nan   # доля пойманных фейков
    out["tnr"] = 1 - pred[y == 0].mean() if (y == 0).any() else np.nan  # доля верно принятых real
    return out

@torch.no_grad()
def predict(model, df):
    model.eval(); out = []
    for x, _ in loader(df, eval_tf):
        with torch.autocast(DEV, enabled=(DEV == "cuda")):
            out.append(torch.sigmoid(model(x.to(DEV)).float().squeeze(1)).cpu())
    return torch.cat(out).numpy()

def train(model, name, epochs, lr):
    model.to(DEV)
    pos_w = torch.tensor([(train_df.y == 0).sum() / max((train_df.y == 1).sum(), 1)], device=DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(DEV, enabled=(DEV == "cuda"))
    tl = loader(train_df, train_tf, shuffle=True)
    best_auc, best_state, log = -1, None, []
    for ep in range(epochs):
        model.train(); tot = 0; t0 = time.time()
        for x, y in tqdm(tl, desc=f"{name} ep{ep+1}/{epochs}", leave=False):
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEV, enabled=(DEV == "cuda")):
                loss = crit(model(x).squeeze(1).float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y)
        sched.step()
        vm = metrics(val_df.y, predict(model, val_df))
        log.append(dict(epoch=ep + 1, train_loss=tot / len(train_df), val_auc=vm["auc"], val_acc=vm["acc"], sec=time.time() - t0))
        print(f"{name} ep{ep+1}: loss={tot/len(train_df):.4f}  val_auc={vm['auc']:.4f}  val_acc={vm['acc']:.4f}  ({time.time()-t0:.0f}s)")
        if vm["auc"] > best_auc:
            best_auc, best_state = vm["auc"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    pd.DataFrame(log).to_csv(f"{CFG['RES']}/trainlog_{name}.csv", index=False)
    torch.save(best_state, f"{CFG['RES']}/models/{name}.pt")
    return model

PRED = {}   # name -> {set_name: probs}

# предсказания каждой модели сразу сохраняем на диск (и в Drive), чтобы при обрыве сессии не считать заново
def save_pred(name):
    np.savez(f"{CFG['RES']}/pred_{name}.npz", **PRED[name])
    try:
        if os.path.isdir("/content/drive/MyDrive"):
            os.makedirs("/content/drive/MyDrive/deepfake_paper/results_partial", exist_ok=True)
            shutil.copy(f"{CFG['RES']}/pred_{name}.npz", "/content/drive/MyDrive/deepfake_paper/results_partial/")
    except Exception as e:
        print("Drive copy skipped:", e)

def load_pred(name):
    for p in (f"{CFG['RES']}/pred_{name}.npz", f"/content/drive/MyDrive/deepfake_paper/results_partial/pred_{name}.npz"):
        if os.path.exists(p):
            z = np.load(p); PRED[name] = {k: z[k] for k in z.files}; print(f"{name}: предсказания загружены из {p}"); return True
    return False

## 3. Детектор 1 — SimpleCNN с нуля

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
        self.f = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.h = nn.Sequential(nn.Dropout(0.3), nn.Linear(256, 1))
    def forward(self, x): return self.h(self.f(x))

if not load_pred("SimpleCNN"):
    cnn = train(SimpleCNN(), "simple_cnn", CFG["EPOCHS_CNN"], CFG["LR_CNN"])
    PRED["SimpleCNN"] = {k: predict(cnn, v) for k, v in EVAL.items()}
    save_pred("SimpleCNN"); del cnn; torch.cuda.empty_cache()

## 4. Детектор 2 — ResNet-50 (ImageNet), дообучение целиком

In [ ]:
if not load_pred("ResNet-50"):
    rn = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    rn.fc = nn.Linear(rn.fc.in_features, 1)
    rn = train(rn, "resnet50", CFG["EPOCHS_RESNET"], CFG["LR_RESNET"])
    PRED["ResNet-50"] = {k: predict(rn, v) for k, v in EVAL.items()}
    save_pred("ResNet-50"); del rn; torch.cuda.empty_cache()

## 5. Детектор 3 — CLIP ViT-L/14 (заморожен) + логистическая регрессия
Признаки извлекаем один раз для всех 5000 картинок и кэшируем. Классификатор обучаем на train, параметр C выбираем по val.

In [ ]:
from transformers import CLIPModel, CLIPImageProcessor
clip = CLIPModel.from_pretrained(CFG["CLIP_ID"], torch_dtype=CLIP_DTYPE).to(DEV).eval()
proc = CLIPImageProcessor.from_pretrained(CFG["CLIP_ID"])

@torch.no_grad()
def clip_feats(df):
    feats = []
    for i in tqdm(range(0, len(df), CFG["BS"]), desc="CLIP feats", leave=False):
        ims = [Image.open(p).convert("RGB").resize((256, 256)) for p in df.path.iloc[i:i + CFG["BS"]]]   # тот же контроль разрешения
        px = proc(images=ims, return_tensors="pt")["pixel_values"].to(DEV, CLIP_DTYPE)
        f = clip.get_image_features(pixel_values=px)
        feats.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu())
    return torch.cat(feats).numpy()

feat_path = f"{CFG['RES']}/clip_feats.npz"
if os.path.exists(feat_path):
    npz = np.load(feat_path); F = {k: npz[k] for k in npz.files}
else:
    F = {"train": clip_feats(train_df), "val": clip_feats(val_df), **{k: clip_feats(v) for k, v in EVAL.items()}}
    np.savez(feat_path, **F)
del clip; torch.cuda.empty_cache()

sc = StandardScaler().fit(F["train"])
best = None
for C in [0.01, 0.1, 1.0, 10.0]:
    lr_ = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(sc.transform(F["train"]), train_df.y)
    auc = roc_auc_score(val_df.y, lr_.predict_proba(sc.transform(F["val"]))[:, 1])
    print(f"C={C}: val_auc={auc:.4f}")
    if best is None or auc > best[0]: best = (auc, C, lr_)
_, bestC, clf = best
print("выбрано C =", bestC)
PRED["CLIP+LR"] = {k: clf.predict_proba(sc.transform(F[k]))[:, 1] for k in EVAL}
save_pred("CLIP+LR")

## 6. Сводные таблицы

In [ ]:
rows = []
for m, d in PRED.items():
    for s, p in d.items():
        rows.append(dict(model=m, test=s, **metrics(EVAL[s].y, p)))
        if s == "test_own":   # отдельно по подтипам наших фейков (доля пойманных)
            df = EVAL[s]
            for ft in ("t2i", "i2i"):
                mk = (df.fake_type == ft).values
                rows.append(dict(model=m, test=f"test_own/{ft}", n=int(mk.sum()), acc=np.nan, auc=np.nan, ap=np.nan, tpr=float((p[mk] >= 0.5).mean()), tnr=np.nan))
M = pd.DataFrame(rows)
M.to_csv(f"{CFG['RES']}/metrics.csv", index=False)
print(M.round(3).to_string(index=False))

# по генераторам: доля пойманных фейков каждого генератора (и real по источникам — доля верно принятых)
rows = []
for m, d in PRED.items():
    for s, p in d.items():
        df = EVAL[s].copy(); df["p"] = p
        for g, grp in df.groupby(["y", "generator", "source"]):
            y, gen, src = g
            rows.append(dict(model=m, test=s, kind="fake" if y == 1 else "real", generator=gen if y == 1 else src,
                             held_out=int(grp.held_out.max()), n=len(grp),
                             rate=float((grp.p >= 0.5).mean()) if y == 1 else float((grp.p < 0.5).mean())))
G = pd.DataFrame(rows).drop_duplicates(["model", "generator", "kind", "test"])
G.to_csv(f"{CFG['RES']}/per_generator.csv", index=False)
print("\nДоля пойманных фейков по генераторам:")
print(G[G.kind == "fake"].pivot_table(index=["test", "held_out", "generator"], columns="model", values="rate").round(3).to_string())

# все предсказания — пригодятся для разбора ошибок и для статьи
P = []
for m, d in PRED.items():
    for s, p in d.items():
        df = EVAL[s][["id", "filename", "label", "origin", "generator", "fake_type", "source"]].copy()
        df["model"], df["test"], df["prob_fake"] = m, s, p
        P.append(df)
pd.concat(P).to_csv(f"{CFG['RES']}/predictions.csv", index=False)

## 7. Графики: главный график, матрицы ошибок, примеры ошибок

In [ ]:
sets = ["test_seen", "test_unseen", "test_own"]; names = list(PRED)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ["auc", "acc"], ["ROC-AUC", "Accuracy (thr = 0.5)"]):
    w = 0.8 / len(names)
    for i, m in enumerate(names):
        vals = [M[(M.model == m) & (M.test == s)][metric].values[0] for s in sets]
        ax.bar(np.arange(len(sets)) + i * w, vals, w, label=m)
        for j, v in enumerate(vals): ax.text(j + i * w, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(np.arange(len(sets)) + w * (len(names) - 1) / 2); ax.set_xticklabels(["seen\ngenerators", "unseen\ngenerators", "our set\n(COCO + SDXL-Turbo)"])
    ax.set_ylim(0.4, 1.05); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
axes[0].legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{CFG['RES']}/figs/fig_main_generalization.png", dpi=200); plt.close()

fig, axes = plt.subplots(len(names), len(sets), figsize=(3.2 * len(sets), 3 * len(names)))
for i, m in enumerate(names):
    for j, s in enumerate(sets):
        cm = confusion_matrix(EVAL[s].y, (PRED[m][s] >= 0.5).astype(int), labels=[0, 1])
        ax = axes[i, j]; ax.imshow(cm, cmap="Blues")
        for a in range(2):
            for b in range(2): ax.text(b, a, cm[a, b], ha="center", va="center", color="black" if cm[a, b] < cm.max() / 2 else "white")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["pred real", "pred fake"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["real", "fake"])
        ax.set_title(f"{m} — {s}", fontsize=9)
plt.tight_layout(); plt.savefig(f"{CFG['RES']}/figs/fig_confusion.png", dpi=200); plt.close()

# доля пойманных фейков по генераторам (лучшая по test_own модель)
best_m = M[M.test == "test_own"].sort_values("auc").model.iloc[-1]
g = G[(G.kind == "fake") & (G.model == best_m)].sort_values(["test", "rate"])
plt.figure(figsize=(9, 4.5))
colors = ["#4C72B0" if t == "test_seen" else "#DD8452" if t == "test_unseen" else "#55A868" for t in g.test]
plt.barh(g.generator + "  (" + g.test.str.replace("test_", "") + ")", g.rate, color=colors)
plt.xlabel("fraction of fakes detected (thr = 0.5)"); plt.title(f"{best_m}: detection rate per generator"); plt.xlim(0, 1.05)
plt.tight_layout(); plt.savefig(f"{CFG['RES']}/figs/fig_per_generator.png", dpi=200); plt.close()

# примеры ошибок лучшей модели на нашем наборе: пропущенные фейки и ложные тревоги на real
df = EVAL["test_own"].copy(); df["p"] = PRED[best_m]["test_own"]
fn = df[(df.y == 1) & (df.p < 0.5)].sort_values("p").head(6)
fp = df[(df.y == 0) & (df.p >= 0.5)].sort_values("p", ascending=False).head(6)
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax_row, sub, ttl in zip(axes, [fn, fp], ["missed fake", "false alarm (real)"]):
    for ax, (_, r) in zip(ax_row, sub.iterrows()):
        ax.imshow(Image.open(r.path)); ax.set_title(f"{ttl}\n{r.fake_type if r.y == 1 else 'real'}  p={r.p:.2f}", fontsize=8)
    for ax in ax_row: ax.axis("off")
plt.suptitle(f"{best_m}: errors on our set"); plt.tight_layout()
plt.savefig(f"{CFG['RES']}/figs/fig_errors_own.png", dpi=200); plt.close()

print("лучшая модель на test_own:", best_m)
for f in sorted(os.listdir(f"{CFG['RES']}/figs")): print(" ", f)

## 8. Архив результатов (без весов моделей) → Google Drive

In [ ]:
with zipfile.ZipFile("/content/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(CFG["RES"]):
        for f in fs:
            if not root.endswith("models"):
                z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), CFG["RES"]))
try:
    from google.colab import drive; drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/deepfake_paper", exist_ok=True)
    shutil.copy("/content/results.zip", "/content/drive/MyDrive/deepfake_paper/results.zip")
    print("скопировано: Google Drive / deepfake_paper / results.zip")
except Exception as e:
    print("Drive не подключён, архив в /content/results.zip", e)

from IPython.display import Image as IPImage, display
display(IPImage(f"{CFG['RES']}/figs/fig_main_generalization.png"))